# Notebook AWS — Extraction de features + PCA avec StandardScaler (EMR/PySpark)

# Commencement AWS : Création utilisateur

**nom utilisateur: cli-lylber**

On fournit des autorisations administratives pour S3 et en général. Clé activée.  
Télécharger le csv qui contient la paire de clés.

# Commencement local : pip install awscli

**Configuration AWS**
- On rentre l'ID et la clé secrète pour l'utilisateur cli-lylber  
- On choisit la localisation `eu-west-3` pour avoir les serveurs à Paris : rapidité et protection des données selon la législature européenne  
- Format de sortie par défaut : JSON

# Création d'un bucket S3

- commande powershell : `aws s3 mb s3://lylber-p8-data`
- résultat : `make_bucket: lylber-p8-data`

On se positionne dans le dossier Test et on fait une synchronisation avec le bucket :  
`aws s3 sync . s3://lylber-p8-data/Test`

# Création clés SSH

Les clés SSH sont un moyen sécurisé d'établir une connexion entre deux machines via un réseau non sécurisé.  
Elles sont essentielles pour lancer une machine EMR (qui se repose sur les machines EC2).  
RSA : Recommandé pour une compatibilité maximale.

- `.pem` : OpenSSH, compatible Linux  
- `.ppk` : PuTTY, compatible Windows (celle qu'on va utiliser)

# Autorisation du tunnel SSH dans AWS

Par défaut, AWS bloque l'accès au port 22 (SSH) pour des raisons de sécurité.  
En autorisant le tunnel SSH à travers le port 22 dans le firewall d'AWS, on peut établir une connexion sécurisée et accéder à JupyterHub et aux logs de Spark depuis notre machine locale.

# Configuration du serveur EMR

- Sélectionner : Spark, Hadoop, TensorFlow, JupyterHub (attention à la compatibilité des versions)  
- Configuration de cluster : Flottes d'instances flexibles  
  - Instance maître : `m5.xlarge`  
  - Instance nœud : `m5.xlarge`  
- Activer : *Résilier automatiquement le cluster après le temps d'inactivité*  
- Ajouter une action d'amorçage : installation de packages à tous les niveaux du cluster  
- Ajouter la paire de clés PuTTY  
- Donner les autorisations lecture/écriture S3 aux instances

# Établir le tunnel SSH avec le cluster

- Cliquer sur *Connexion au nœud primaire à l'aide de SSH*, onglet Windows, copier le lien  
- Ouvrir PuTTY, ajouter le lien, charger la clé `.ppk`, accepter  
- Configurer le navigateur pour utiliser le tunnel SSH (proxy SOCKS)

# Connexion JupyterHub (onglet Application) et lancement du notebook

---
# Partie 1 — Extraction de features (MobileNetV2)

In [ ]:
%%configure -f
{"driverMemory": "6000M"}

In [ ]:
#  L'exécution de cette cellule démarre l'application Spark

In [ ]:
%%info

In [ ]:
import pandas as pd
import numpy as np
import io
import os
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split
from pyspark.sql import SparkSession

In [ ]:
PATH        = 's3://nons-bucket-p9-096766144198-eu-west-3-an'
PATH_Data   = PATH + '/data/Test'
PATH_Result = PATH + '/data/Results'
PATH_PCA    = PATH + '/data/PCA'
print('PATH:        ' + PATH +
      '\nPATH_Data:   ' + PATH_Data +
      '\nPATH_Result: ' + PATH_Result +
      '\nPATH_PCA:    ' + PATH_PCA)

In [ ]:
# ============================================================
# MODE D'EXÉCUTION
# TEST_MODE = True  → échantillon limité (rapide, pour valider le pipeline)
# TEST_MODE = False → dataset complet   (run de production)
# ============================================================
TEST_MODE   = True
TEST_SAMPLE = 100   # nombre d'images à utiliser en mode test

In [ ]:
images = spark.read.format('binaryFile') \
    .option('pathGlobFilter', '*.jpg') \
    .option('recursiveFileLookup', 'true') \
    .load(PATH_Data)

In [ ]:
images = images.withColumn('label', element_at(split(images['path'], '/'), -2))
print(images.printSchema())
print(images.select('path', 'label').show(5, False))

In [ ]:
if TEST_MODE:
    images = images.sample(fraction=0.05, seed=42).limit(TEST_SAMPLE)
    print(f'MODE TEST — {images.count()} images chargées (limite : {TEST_SAMPLE})')
else:
    print(f'MODE COMPLET — {images.count()} images chargées')

In [ ]:
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

In [ ]:
brodcast_weights = sc.broadcast(new_model.get_weights())

In [ ]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                      outputs=model.layers[-2].output)
    new_model.set_weights(brodcast_weights.value)
    return new_model

In [ ]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).convert('RGB').resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)


def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    output = [p.flatten() for p in preds]
    return pd.Series(output)


@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    """
    Scalar Iterator pandas UDF wrapping our featurization function.
    Loads the model once and reuses it across batches to amortize overhead.
    """
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

In [ ]:
spark.conf.set('spark.sql.execution.arrow.maxRecordsPerBatch', '1024')

features_df = images.repartition(24).select(
    col('path'),
    col('label'),
    featurize_udf('content').alias('features')
)

features_df.write.mode('overwrite').parquet(PATH_Result)
print('Features écrites dans', PATH_Result)

---
# Partie 2 — PCA avec StandardScaler

Pipeline :
1. `VectorAssembler` : column `features` (array) → `features_vec` (DenseVector)
2. `StandardScaler` : centrage et réduction (`withMean=True`, `withStd=True`) → `features_scaled`
3. PCA exploratoire (`k_max=200`) + recherche du k optimal (seuils 80 / 90 / 95 %)
4. PCA finale avec `k_optimal` (cible 90 %) → `pcaFeatures`

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.linalg import Vectors

In [ ]:
df = pd.read_parquet(PATH_Result, engine='pyarrow')
print(f'Shape                  : {df.shape}')
print(f'Taille vecteur features: {df.loc[0, "features"].shape}')
df.head()

In [ ]:
spark.sparkContext.setLogLevel('ERROR')
spark.conf.set('spark.sql.execution.arrow.maxRecordsPerBatch', '1024')

df['features'] = df['features'].apply(lambda x: Vectors.dense(x))
data_spark = spark.createDataFrame(df)

vecAssembler = VectorAssembler(inputCols=['features'], outputCol='features_vec')
data_spark = vecAssembler.transform(data_spark)
print('VectorAssembler OK')

In [ ]:
scaler = StandardScaler(
    inputCol='features_vec',
    outputCol='features_scaled',
    withMean=True,
    withStd=True
)
scalerModel = scaler.fit(data_spark)
data_scaled = scalerModel.transform(data_spark).cache()
print('StandardScaler OK — features centrées (mean=0) et réduites (std=1)')

In [ ]:
k_max = 200
pca_exp = PCA(k=k_max, inputCol='features_scaled', outputCol='pcaFeatures')
pcaModel_exp = pca_exp.fit(data_scaled)
cumValues = pcaModel_exp.explainedVariance.cumsum()

print('Recherche du k optimal :')
for seuil in [0.80, 0.90, 0.95]:
    mask = cumValues >= seuil
    if mask.any():
        K = int(np.argmax(mask) + 1)
        print(f'  {int(seuil * 100)}% de variance atteint avec k = {K}')
    else:
        print(f'  {int(seuil * 100)}% non atteint avec k_max={k_max} '
              f'(max = {cumValues[-1] * 100:.1f}%)')

plt.figure(figsize=(10, 6))
plt.plot(range(1, k_max + 1), cumValues, marker='o', linestyle='--', markersize=3)
plt.axhline(y=0.80, color='r', linestyle=':', label='80% variance')
plt.axhline(y=0.90, color='g', linestyle=':', label='90% variance')
plt.axhline(y=0.95, color='b', linestyle=':', label='95% variance')
plt.title('Variance expliquée cumulée par composante PCA (après StandardScaler)')
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance cumulée expliquée')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
seuil_cible = 0.90
mask = cumValues >= seuil_cible
k_optimal = int(np.argmax(mask) + 1) if mask.any() else k_max
print(f'PCA finale avec k = {k_optimal} composantes '
      f'({int(seuil_cible * 100)}% de variance expliquée)')

pca_final = PCA(k=k_optimal, inputCol='features_scaled', outputCol='pcaFeatures')
model_pca = pca_final.fit(data_scaled)
result = model_pca.transform(data_scaled)

result = result.drop('features_vec', 'features_scaled')
result.show(5)

In [ ]:
result.write.mode('overwrite').parquet(PATH_PCA)
print('Résultat PCA écrit dans', PATH_PCA)

In [ ]:
df_pca = spark.read.parquet(PATH_PCA).toPandas()
print(f'Shape final : {df_pca.shape}')
print(f'Colonnes    : {list(df_pca.columns)}')
df_pca.head()